In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment Ready!")

Environment Ready!


In [2]:
# 1. Load the new dark store dataset
df = pd.read_csv('Hyper_Realistic_Dark_Store_Weekly_Data.csv')

In [3]:
# 2. Convert Dates
df['Week_Start_Date'] = pd.to_datetime(df['Week_Start_Date'], dayfirst=True)

In [4]:
# 3. Extract Time Features
df['Month'] = df['Week_Start_Date'].dt.month
df['Week_Number'] = df['Week_Start_Date'].dt.isocalendar().week

print(f"Dataset Loaded: {df.shape[0]} rows across {df['store_id'].nunique()} unique stores.")

Dataset Loaded: 84315 rows across 803 unique stores.


In [5]:
df

,store_id,company,city,zone,latitude,longitude,State,Week_Start_Date,Week_End_Date,Temperature_Max_Weekly (°C),...,Road_Closures,Local_Events_Festivals,Political_Rallies_Protests,Heat_Stress_Index_Avg,Wind_Hazard_Score_Avg,Flood_Risk_Index_Max,Days_in_Week,AQI_Category,Month,Week_Number
0,BLR_ZEP_001,Zepto,Bangalore,Indiranagar 100ft Road,12.9771,77.6409,Karnataka,2024-01-01,07-01-2024,31.9,...,1,0,0,14.6,2.4,2.4,7,Satisfactory,1,1
1,BLR_ZEP_001,Zepto,Bangalore,Indiranagar 100ft Road,12.9771,77.6409,Karnataka,2024-04-01,07-04-2024,37.5,...,1,1,0,20.9,2.5,2.4,7,Satisfactory,4,14
2,BLR_ZEP_001,Zepto,Bangalore,Indiranagar 100ft Road,12.9771,77.6409,Karnataka,2024-07-01,07-07-2024,30.9,...,0,0,0,21.3,1.9,6.9,7,Satisfactory,7,27
3,BLR_ZEP_001,Zepto,Bangalore,Indiranagar 100ft Road,12.9771,77.6409,Karnataka,2025-09-01,07-09-2025,31.8,...,1,0,1,17.6,2.4,4.0,7,Satisfactory,9,36
4,BLR_ZEP_001,Zepto,Bangalore,Indiranagar 100ft Road,12.9771,77.6409,Karnataka,2025-12-01,07-12-2025,30.0,...,0,0,0,16.6,1.1,2.1,7,Moderate,12,49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84310,DDN_INS_007,Instamart,Dehradun,GMS Road,30.2660,78.0780,Delhi,2025-12-29,04-01-2026,23.2,...,0,0,0,6.9,1.8,2.4,3,Poor,12,1
84311,DDN_INS_007,Instamart,Dehradun,GMS Road,30.2660,78.0780,Delhi,2025-06-30,06-07-2025,43.8,...,0,0,0,30.8,2.2,3.8,7,Moderate,6,27
84312,DDN_INS_007,Instamart,Dehradun,GMS Road,30.2660,78.0780,Delhi,2024-09-30,06-10-2024,36.2,...,1,1,1,20.6,2.4,2.9,7,Moderate,9,40
84313,DDN_INS_007,Instamart,Dehradun,GMS Road,30.2660,78.0780,Delhi,2024-12-30,05-01-2025,25.2,...,1,0,0,3.9,2.5,1.6,7,Poor,12,1


In [6]:
def calculate_store_risk(row):
    # --- Category A: Climate Factors (Max 50 pts) ---
    # Temp: Deviation from 25°C (15 pts)
    temp_r = min(15, abs(row['Temperature_Avg_Weekly (°C)'] - 25))
    # Rainfall: 15 pts if > 50mm
    rain_r = min(15, (row['Rainfall_Total (mm)'] / 50) * 15)
    # AQI: 10 pts if > 400
    aqi_r = min(10, (row['AQI_Avg'] / 400) * 10)
    # Humidity: 10 pts if high
    hum_r = min(10, max(0, row['Humidity_Avg (%)'] - 60) / 4)
    
    # --- Category B: Social/Local Disruptions (Max 50 pts) ---
    # These are binary (0 or 1) in your dataset
    disruption_r = (row['Road_Closures'] * 20) + \
                   (row['Local_Events_Festivals'] * 15) + \
                   (row['Political_Rallies_Protests'] * 15)
    
    total_score = temp_r + rain_r + aqi_r + hum_r + disruption_r
    return round(min(100, total_score), 2)

In [7]:
# Apply the logic
df['Risk_Score'] = df.apply(calculate_store_risk, axis=1)

print("Risk Scores generated for all stores and timeframes!")
df[['store_id', 'company', 'city', 'Risk_Score']].head()

Risk Scores generated for all stores and timeframes!


,store_id,company,city,Risk_Score
0,BLR_ZEP_001,Zepto,Bangalore,24.17
1,BLR_ZEP_001,Zepto,Bangalore,40.65
2,BLR_ZEP_001,Zepto,Bangalore,21.28
3,BLR_ZEP_001,Zepto,Bangalore,54.64
4,BLR_ZEP_001,Zepto,Bangalore,7.46


In [8]:
# Initialize Encoders
store_enc = LabelEncoder()
city_enc = LabelEncoder()
zone_enc = LabelEncoder()
comp_enc = LabelEncoder()

In [9]:
# Fit and Transform
df['Store_Code'] = store_enc.fit_transform(df['store_id'])
df['City_Code'] = city_enc.fit_transform(df['city'])
df['Zone_Code'] = zone_enc.fit_transform(df['zone'])
df['Company_Code'] = comp_enc.fit_transform(df['company'])

In [10]:
# Save Encoders for Backend use later
joblib.dump(store_enc, 'store_encoder.pkl')
joblib.dump(city_enc, 'city_encoder.pkl')
joblib.dump(zone_enc, 'zone_encoder.pkl')
joblib.dump(comp_enc, 'company_encoder.pkl')

print("Encoders saved successfully.")

Encoders saved successfully.


In [11]:
# 1. Define inputs for the model
# We include Store, City, Zone, Month, and Week
features = ['Store_Code', 'City_Code', 'Zone_Code', 'Month', 'Week_Number']
X = df[features]
y = df['Risk_Score']

In [12]:
# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

In [13]:
# 3. Train XGBoost
model = xgb.XGBRegressor(
    n_estimators=300, 
    learning_rate=0.1, 
    max_depth=8, 
    objective='reg:squarederror'
)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [14]:
# 4. Save the Final Model
model.save_model("quickshield_store_risk_model.json")

print(f"Model Trained! Test R2 Score: {round(model.score(X_test, y_test)*100, 2)}%")

Model Trained! Test R2 Score: 62.99%


In [15]:
# Save the final calculated risk scores to a CSV
df.to_csv('QuickShield_Store_Risk_Registry.csv', index=False)

In [16]:
# Save Metadata for the teammate
metadata = {
    "stores": dict(zip(store_enc.classes_, store_enc.transform(store_enc.classes_).astype(int).tolist())),
    "zones": dict(zip(zone_enc.classes_, zone_enc.transform(zone_enc.classes_).astype(int).tolist())),
    "features": features
}

with open('store_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

In [17]:
%%writefile predict_store_risk.py
import sys
import pandas as pd
import xgboost as xgb
import joblib
from datetime import datetime

def predict_risk(store_id, city, zone, date_str):
    try:
        # 1. Load Model and Encoders
        model = xgb.XGBRegressor()
        model.load_model("quickshield_store_risk_model.json")
        
        s_enc = joblib.load('store_encoder.pkl')
        c_enc = joblib.load('city_encoder.pkl')
        z_enc = joblib.load('zone_encoder.pkl')

        # 2. Process Inputs
        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
        month = date_obj.month
        week = date_obj.isocalendar()[1]
        
        # Convert text to IDs
        s_id = s_enc.transform([store_id])[0]
        c_id = c_enc.transform([city])[0]
        z_id = z_enc.transform([zone])[0]

        # 3. Predict
        input_df = pd.DataFrame([[s_id, c_id, z_id, month, week]], 
                                columns=['Store_Code', 'City_Code', 'Zone_Code', 'Month', 'Week_Number'])
        
        score = model.predict(input_df)[0]
        return round(float(score), 2)

    except Exception as e:
        return f"Error: {str(e)}"

if __name__ == "__main__":
    # Expects: python predict_store_risk.py "BLR_ZEP_001" "Bangalore" "Indiranagar" "2026-04-02"
    if len(sys.argv) == 5:
        print(predict_risk(sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4]))

Writing predict_store_risk.py
